In [30]:
import pandas as pd
import load_TU_data
from otp_client import get_all_routes_for_mode, load_all_candidates
from find_similar_trip import find_similar_trip
from otp_utils import has_invalid_route_name, resolve_route_short_names
import importlib

In [6]:
print("Loading TU data...")
tu_session, tu_tur, tu_deltur = load_TU_data.load_tu(
    data_dir="/home/simpal/O/TU_Rejseplan/Data/TU/",
    session_file="tu_session_secret_2015_2025.xlsx",
    tur_file="tu_tur_secret_2015_2025.xlsx",
    deltur_file="tu_deltur_2015_2025.xlsx"
)
print("TU data loaded")

Loading TU data...
TU data loaded


In [50]:
tu_tur["DiaryDate"]

4         16436
6         16436
10        16436
17        16436
46        16437
          ...  
322023    20450
322034    20095
322035    20095
322036    20095
322037    20095
Name: DiaryDate, Length: 18286, dtype: int64

In [51]:
tu_tur

,TurId,SessionId,TurNr,TripCount,DepartHH,DepartMM,DepartMSM,ArrivalHH,ArrivalMM,ArrivalMSM,...,WorkplE,WorkplN,Startstedadre,Startstedadrn,date,date_str,depart_dt,depart_dt_str,arrival_dt,arrival_dt_str
4,2091577,337215,1,1.0,17.0,30.0,1050.0,17.0,53.0,1073.0,...,553742.0,6320078.0,556701.0,6322429.0,2015-01-01,2015-01-01,2015-01-01 17:30:00+01:00,2015-01-01T17:30:00+0100,2015-01-01 17:53:00+01:00,2015-01-01T17:53:00+0100
6,2091579,337218,1,1.0,13.0,0.0,780.0,14.0,14.0,854.0,...,NaN,NaN,699455.0,6186647.0,2015-01-01,2015-01-01,2015-01-01 13:00:00+01:00,2015-01-01T13:00:00+0100,2015-01-01 14:14:00+01:00,2015-01-01T14:14:00+0100
10,2091583,337220,1,1.0,3.0,50.0,230.0,4.0,48.0,288.0,...,705497.0,6165196.0,700465.0,6166135.0,2015-01-01,2015-01-01,2015-01-01 03:50:00+01:00,2015-01-01T03:50:00+0100,2015-01-01 04:48:00+01:00,2015-01-01T04:48:00+0100
17,2091590,337222,2,1.0,13.0,20.0,800.0,16.0,37.0,997.0,...,720230.0,6211603.0,601281.0,6129550.0,2015-01-01,2015-01-01,2015-01-01 13:20:00+01:00,2015-01-01T13:20:00+0100,2015-01-01 16:37:00+01:00,2015-01-01T16:37:00+0100
46,2091628,337245,1,1.0,18.0,0.0,1080.0,18.0,19.0,1099.0,...,555417.0,6192177.0,573670.0,6222834.0,2015-01-02,2015-01-02,2015-01-02 18:00:00+01:00,2015-01-02T18:00:00+0100,2015-01-02 18:19:00+01:00,2015-01-02T18:19:00+0100
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
322023,2768511,548642,4,1.0,23.0,35.0,1415.0,24.0,29.0,1469.0,...,546692.0,6158260.0,544504.0,6161710.0,2025-12-28,2025-12-28,2025-12-28 23:35:00+01:00,2025-12-28T23:35:00+0100,2025-12-29 00:29:00+01:00,2025-12-29T00:29:00+0100
322034,2768715,548777,1,1.0,7.0,30.0,450.0,7.0,52.0,472.0,...,725084.0,6176370.0,722312.0,6176885.0,2025-01-07,2025-01-07,2025-01-07 07:30:00+01:00,2025-01-07T07:30:00+0100,2025-01-07 07:52:00+01:00,2025-01-07T07:52:00+0100
322035,2768717,548777,2,1.0,8.0,5.0,485.0,8.0,24.0,504.0,...,725084.0,6176370.0,722312.0,6176885.0,2025-01-07,2025-01-07,2025-01-07 08:05:00+01:00,2025-01-07T08:05:00+0100,2025-01-07 08:24:00+01:00,2025-01-07T08:24:00+0100
322036,2768718,548777,3,1.0,15.0,30.0,930.0,15.0,55.0,955.0,...,725084.0,6176370.0,722312.0,6176885.0,2025-01-07,2025-01-07,2025-01-07 15:30:00+01:00,2025-01-07T15:30:00+0100,2025-01-07 15:55:00+01:00,2025-01-07T15:55:00+0100


In [52]:
period = (tu_tur.loc[tu_tur["DiaryYear"] == 2024, "DiaryDate"].min(), tu_tur.loc[tu_tur["DiaryYear"] == 2024, "DiaryDate"].max())
period

(19725, 20086)

In [7]:
#Configurartion
mode_map = {
    #TU: OTP
    31: "BUS",
    32: "S_TRAIN",
    33: "RAIL",
    34: "SUBWAY",
    37: "TRAM",
    41: "FERRY",
    35: "BUS"
}
otp_url = "http://localhost:8080/otp/gtfs/v1"
search_window = "PT30M"

print(f"Search window: {search_window}")

Search window: PT30M


In [8]:
# RAIL, TRAM and SUBWAY are missing route name in TU.
# So taking all routes for these modes. Which will be used when modes
# that do include route name in TU only can access those routes, but for
# those that do not, all routes will be used.
otp_mode_routes_cache = {mode: get_all_routes_for_mode(mode) for mode in ["RAIL", "TRAM", "SUBWAY", "FERRY"]}

In [9]:
tu_tur = tu_tur[tu_tur["PtPrimMode"].isin([31, 32, 33, 34, 37, 41])]
#tu_tur = tu_tur[(tu_tur["DiaryYear"] == 2024) & (tu_tur["DiaryMonth"] == 6)]

In [21]:

i_TurId = 2648141
tu_tur_row = tu_tur[tu_tur["TurId"] == i_TurId].iloc[0]
tu_deltur_sub = tu_deltur.loc[tu_deltur["TurId"] == i_TurId]

In [23]:
tu_deltur_sub

,SessionId,TurId,Delturnr,StageMode,ModeGroup,StageDrivPass,StageLength,StageWaitMin,StageStartMsm,StageDurationMin,Route,FromStation,ToStation,n_deltur
303109,505753,2648141,1,1,1,NaN,1.0,NaN,410.0,13.0,NaN,NaN,NaN,5
303110,505753,2648141,2,31,120,2.0,4.8,3.0,426.0,15.0,6A,NaN,NaN,5
303111,505753,2648141,3,1,1,NaN,0.1,NaN,441.0,2.0,NaN,NaN,NaN,5
303112,505753,2648141,4,34,110,NaN,7.3,8.0,451.0,12.0,M1,Nørreport,Ørestad,5
303113,505753,2648141,5,1,1,NaN,0.5,NaN,463.0,7.0,NaN,NaN,NaN,5


In [39]:
import otp_utils
importlib.reload(otp_utils)
from otp_utils import resolve_route_short_names

tu_deltur_sub_print_col = ["StageMode", "StageLength", "StageWaitMin", "StageDurationMin", "Route", "FromStation", "ToStation"]
print(f"tu_deltur_sub: {tu_deltur_sub[tu_deltur_sub_print_col]}")
route_names, route_names_ext, modes_json, modes_list = resolve_route_short_names(tu_deltur_sub,
                                                                mode_map,
                                                                otp_mode_routes_cache)

tu_deltur_sub:         StageMode  StageLength  StageWaitMin  StageDurationMin Route  \
303109          1          1.0           NaN              13.0   NaN   
303110         31          4.8           3.0              15.0    6A   
303111          1          0.1           NaN               2.0   NaN   
303112         34          7.3           8.0              12.0    M1   
303113          1          0.5           NaN               7.0   NaN   

       FromStation ToStation  
303109         NaN       NaN  
303110         NaN       NaN  
303111         NaN       NaN  
303112   Nørreport   Ørestad  
303113         NaN       NaN  


In [40]:
if not modes_json:
    print(f"No valid public transport modes found for TurId: {i_TurId}")
print(f"modes_json: {modes_json}")
if any(mode in ["BUS", "S_TRAIN"] for mode in modes_list) and not route_names:
    print(f"No valid BUS and S_TRAIN route found for TurId: {i_TurId}")
if has_invalid_route_name(route_names):
    print(f"Invalid route name: {route_names}")

print(f"route_name: {route_names}")
print(f"route_name_ext: {route_names_ext}")

modes_json: [{'mode': 'BUS'}, {'mode': 'SUBWAY'}]
route_name: ['6A']
route_name_ext: ['M3', 'M2', 'M4', 'M1', '6A']


In [43]:
modes_list

['BUS', 'SUBWAY']

In [44]:
is_bus_s_train = any(mode in ["BUS", "S_TRAIN"] for mode in modes_list)
is_rail_tram_subway_ferry = any(mode in ["RAIL", "SUBWAY", "TRAM", "FERRY"] for mode in modes_list)

if is_bus_s_train and is_rail_tram_subway_ferry:
    otp_candidates_df = load_all_candidates(
        tu_tur_row=tu_tur_row,
        modes_json=modes_json,
        route_short_name=route_names_ext, #extented
        search_window=search_window,
        otp_url=otp_url
    )
elif is_bus_s_train:
    otp_candidates_df = load_all_candidates(
        tu_tur_row=tu_tur_row,
        modes_json=modes_json,
        route_short_name=route_names, #not extented
        search_window=search_window,
        otp_url=otp_url
    )
elif is_rail_tram_subway_ferry:
    otp_candidates_df = load_all_candidates(
        tu_tur_row=tu_tur_row,
        modes_json=modes_json,
        #route_short_name=route_names,
        search_window=search_window,
        otp_url=otp_url
    )
otp_candidates_df

,start,end,system_notice_tag,system_notice_text,iteration_id,leg_id,mode,route_short_name,distance_km,duration_min,generalized_cost,start_time,end_time,from,to,leg_geometry,start_dt
0,2024-02-09T05:53:30+01:00,2024-02-09T06:38:47+01:00,[],[],-22,0,WALK,None,0.031,0,74,1707454410000,1707454440000,Origin,Bispebjerg Torv (Tagensvej),"[(55.71588, 12.53181), (55.71593, 12.53171), (...",2024-02-09 05:53:30+01:00
1,2024-02-09T05:53:30+01:00,2024-02-09T06:38:47+01:00,[],[],-22,1,BUS,6A,4.879,15,1500,1707454440000,1707455340000,Bispebjerg Torv (Tagensvej),Nørreport St. (Nørre Voldgade),"[(55.71592, 12.53195), (55.71592, 12.53195), (...",2024-02-09 05:53:30+01:00
2,2024-02-09T05:53:30+01:00,2024-02-09T06:38:47+01:00,[],[],-22,2,WALK,None,0.135,4,381,1707455340000,1707455609000,Nørreport St. (Nørre Voldgade),Nørreport St. (Metro),"[(55.68404, 12.57262), (55.68403, 12.57264), (...",2024-02-09 05:53:30+01:00
3,2024-02-09T05:53:30+01:00,2024-02-09T06:38:47+01:00,[],[],-22,3,SUBWAY,M1,7.241,12,1591,1707455880000,1707456600000,Nørreport St. (Metro),Ørestad St. (Metro),"[(55.68385, 12.57106), (55.68365, 12.57158), (...",2024-02-09 05:53:30+01:00
4,2024-02-09T05:53:30+01:00,2024-02-09T06:38:47+01:00,[],[],-22,4,WALK,None,0.447,9,991,1707456600000,1707457127000,Ørestad St. (Metro),Destination,"[(55.62905, 12.57938), (55.62904, 12.57943), (...",2024-02-09 05:53:30+01:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
191,2024-02-09T07:50:30+01:00,2024-02-09T08:40:47+01:00,[outside-search-window],[This itinerary is marked as deleted by the ou...,17,0,WALK,None,0.031,0,74,1707461430000,1707461460000,Origin,Bispebjerg Torv (Tagensvej),"[(55.71588, 12.53181), (55.71593, 12.53171), (...",2024-02-09 07:50:30+01:00
192,2024-02-09T07:50:30+01:00,2024-02-09T08:40:47+01:00,[outside-search-window],[This itinerary is marked as deleted by the ou...,17,1,BUS,6A,4.879,20,1800,1707461460000,1707462660000,Bispebjerg Torv (Tagensvej),Nørreport St. (Nørre Voldgade),"[(55.71592, 12.53195), (55.71592, 12.53195), (...",2024-02-09 07:50:30+01:00
193,2024-02-09T07:50:30+01:00,2024-02-09T08:40:47+01:00,[outside-search-window],[This itinerary is marked as deleted by the ou...,17,2,WALK,None,0.135,4,381,1707462660000,1707462929000,Nørreport St. (Nørre Voldgade),Nørreport St. (Metro),"[(55.68404, 12.57262), (55.68403, 12.57264), (...",2024-02-09 07:50:30+01:00
194,2024-02-09T07:50:30+01:00,2024-02-09T08:40:47+01:00,[outside-search-window],[This itinerary is marked as deleted by the ou...,17,3,SUBWAY,M1,7.241,12,1591,1707463200000,1707463920000,Nørreport St. (Metro),Ørestad St. (Metro),"[(55.68385, 12.57106), (55.68365, 12.57158), (...",2024-02-09 07:50:30+01:00


In [ ]:

if otp_candidates_df.empty:
    print(f"No OTP trips found for TurId: {i_TurId}")
    continue

#TODO: Currently only considering trips that include all routes in route_short_name.
#      But for RAIL, TRAM and SUBWAY, route_short_name includes all their routes.
#      Only check for BUS and S_TRAIN (maybe not even S_TRAIN). From/ToStation makes
#      sure other modes are handled "correctly".
required_routes = set(route_names)
iteration_ids_with_all_routes = (
    otp_candidates_df.groupby("iteration_id")["route_short_name"]
    .apply(lambda routes: required_routes.issubset(set(routes.astype(str))))
)

otp_candidates_df = otp_candidates_df[
    otp_candidates_df["iteration_id"].isin(
        iteration_ids_with_all_routes[iteration_ids_with_all_routes].index
    )
].reset_index(drop=True)
if otp_candidates_df.empty:
    print(f"No OTP trips include all routes {route_names} for TurId: {i_TurId}")
    continue

time_based_match = find_similar_trip(tu_tur_row, otp_candidates_df, arrival_dev_weight=1)
if time_based_match is None:
    print(f"No best trip found for TurId: {i_TurId}")
    continue
time_based_match["TurId"] = i_TurId


